# Day 2 · DockLab: pockets, docking and refinement

In this practical you will choose a plausible binding pocket, set a docking box, compare focused and blind docking, examine two contrasting ligands, and minimise one selected pose. You will use the docking scores and interaction diagrams together when discussing the results.

Start with the built-in **Demo** example. It uses c-Abl, imatinib as the expected binder, and caffeine as a simple comparison decoy. The optional final activity uses SwissSimilarity to find molecules related to a lead compound. See the [Day 2 practical guide](https://judeyu99.github.io/molecular-discovery-practical-students/day2.html) for the timetable and questions to record.

Run the numbered cells from top to bottom using a Colab CPU runtime. The code is hidden behind named steps. Use the menus to change a 3D representation and the PNG button to save a view after rotating or zooming.

**Sample data:** [download the Day 2 sample pack](https://judeyu99.github.io/molecular-discovery-practical-students/examples/day2_sample_data.zip) if you want to upload the prepared receptor and ligand files yourself. The pack also contains the focused box settings, activity notes and a results worksheet.

For your own system, provide one prepared protein `.pdb` and one ligand as `.sdf`, `.mol2`, `.mol`, `.smi`, SMILES text, or a sketch. Check protonation, stereochemistry, missing atoms, waters, metals and cofactors before interpreting a result. A Vina score is not a measured binding affinity.

**Practical design:** Yu Zhu and Taufiq Rahman · Department of Pharmacology, University of Cambridge. Found an error? Contact Yu Zhu at [yz876@cam.ac.uk](mailto:yz876@cam.ac.uk).

In [ ]:
# @title Step 1. Install tools { display-mode: "form" }
%pip -q install 'meeko==0.7.1' 'rdkit>=2024.9' 'gemmi>=0.6' 'py3Dmol>=2.0' 'ipywidgets>=8,<9' 'ipyketcher==0.1.0'

In [ ]:
# @title Step 2. Start DockLab { display-mode: "form" }
from pathlib import Path
import csv, json, platform, re, subprocess, urllib.request, zipfile
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from IPython.display import display, HTML, Markdown
import py3Dmol

WORK = Path('/content/docklab')
WORK.mkdir(exist_ok=True)

# Vina's pip package can fall back to a source build on newer Colab Python.
# Use the official, precompiled Linux executable instead.
machine = platform.machine().lower()
if platform.system() != 'Linux' or machine not in ('x86_64', 'aarch64'):
    raise RuntimeError('This notebook needs a Linux Colab runtime (x86_64 or aarch64).')
VINA_BIN = WORK / 'vina_1.2.7'
asset = f'vina_1.2.7_linux_{machine}'
url = f'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/{asset}'
if not VINA_BIN.exists():
    urllib.request.urlretrieve(url, VINA_BIN)
VINA_BIN.chmod(0o755)
check = subprocess.run([str(VINA_BIN), '--version'], text=True, capture_output=True, check=True)
print('AutoDock Vina:', (check.stdout or check.stderr).strip())

def run(*args):
    result = subprocess.run([str(x) for x in args], cwd=WORK, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(map(str,args))}\n{result.stdout}\n{result.stderr}")
    return result

def add_residue_hover(view, model_index=0):
    # Protein atoms are hover targets; no residue-marker spheres are needed.
    view.startjs += f'''viewer_UNIQUEID.getModel({model_index}).setHoverable(
      {{}}, true,
      function(atom, viewer) {{
        if (!atom.residueLabel) {{
          var chain = atom.chain || '(blank)';
          var insertion = atom.icode || '';
          atom.residueLabel = viewer.addLabel(
            atom.resn + ' ' + atom.resi + insertion + ' · chain ' + chain + ' · atom ' + atom.atom,
            {{position: atom, backgroundColor: 'white', fontColor: '#172b3a',
              borderColor: '#2878b5', borderThickness: 1, fontSize: 14, inFront: true}});
        }}
      }},
      function(atom, viewer) {{
        if (atom.residueLabel) {{
          viewer.removeLabel(atom.residueLabel);
          delete atom.residueLabel;
        }}
      }});
'''

def add_pocket_sticks(view, protein_pdb, ligand_mol, cutoff_A=5.0):
    # Highlight complete residues with a heavy atom near the docked ligand.
    conf = ligand_mol.GetConformer()
    ligand_xyz = np.array([list(conf.GetAtomPosition(a.GetIdx()))
                           for a in ligand_mol.GetAtoms() if a.GetSymbol() != 'H'])
    pocket = set()
    for line in protein_pdb.splitlines():
        if not line.startswith('ATOM  ') or line[76:78].strip().upper() == 'H':
            continue
        xyz = np.array([float(line[i:i+8]) for i in (30, 38, 46)])
        if np.linalg.norm(ligand_xyz - xyz, axis=1).min() <= cutoff_A:
            pocket.add((line[21], int(line[22:26])))
    for chain, resi in sorted(pocket):
        view.addStyle({'model': 0, 'chain': chain, 'resi': resi},
                      {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.15}})
    return len(pocket)

def style_protein(view, choice, protein_pdb, ligand_mol):
    if choice in ('Cartoon + pocket', 'Cartoon'):
        view.setStyle({'model': 0}, {'cartoon': {'color': '#aab8c5'}})
        if choice == 'Cartoon + pocket':
            add_pocket_sticks(view, protein_pdb, ligand_mol)
    elif choice == 'Sticks':
        view.setStyle({'model': 0}, {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.14}})
    elif choice == 'Ball and stick':
        view.setStyle({'model': 0}, {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.11},
                                    'sphere': {'colorscheme': 'Jmol', 'radius': 0.24}})
    elif choice == 'Lines':
        view.setStyle({'model': 0}, {'line': {'colorscheme': 'Jmol', 'linewidth': 1.5}})
    else:
        raise ValueError(f'Unknown protein representation: {choice}')

def style_ligand(view, choice):
    if choice == 'Sticks':
        style = {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.19}}
    elif choice == 'Ball and stick':
        style = {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.12},
                 'sphere': {'colorscheme': 'greenCarbon', 'radius': 0.27}}
    elif choice == 'Lines':
        style = {'line': {'colorscheme': 'greenCarbon', 'linewidth': 2}}
    else:
        raise ValueError(f'Unknown ligand representation: {choice}')
    view.setStyle({'model': 1}, style)

def add_snapshot_button(view, filename):
    # The button and viewer share one output frame, so it captures the current camera.
    view.startjs += f'''var canvasBox = document.getElementById('3dmolviewer_UNIQUEID');
var saveButton = document.createElement('button');
saveButton.textContent = 'Save current view as PNG';
saveButton.style.cssText = 'display:block;margin:8px 0;padding:9px 14px;background:#145a82;color:white;border:0;border-radius:6px;cursor:pointer;font-weight:600';
saveButton.onclick = function() {{
  viewer_UNIQUEID.render();
  var imageLink = document.createElement('a');
  imageLink.href = viewer_UNIQUEID.pngURI();
  imageLink.download = {json.dumps(filename)};
  document.body.appendChild(imageLink);
  imageLink.click();
  imageLink.remove();
}};
canvasBox.parentNode.insertBefore(saveButton, canvasBox);
'''

def flexible_protein_from_pose(pose_block, protein_path):
    # Transfer optimised side-chain coordinates to the original protein PDB.
    moved = {}
    in_residue = False
    for line in pose_block.splitlines():
        if line.startswith('BEGIN_RES'): in_residue = True
        elif line.startswith('END_RES'): in_residue = False
        elif in_residue and line.startswith('ATOM  '):
            key = (line[21], line[22:27].strip(), line[12:16].strip())
            moved[key] = line[30:54]
    if not moved: raise ValueError('No flexible side-chain atoms were found in the selected pose')
    flexible_ids = {(chain, resid) for chain, resid, atom in moved}
    output = []
    for line in protein_path.read_text().splitlines():
        if not line.startswith(('ATOM  ', 'TER')): continue
        if line.startswith('ATOM  '):
            key = (line[21], line[22:27].strip(), line[12:16].strip())
            atom_name = key[2].upper()
            # Original side-chain hydrogen positions are stale after rotation.
            if ((key[0], key[1]) in flexible_ids and key not in moved
                and (line[76:78].strip().upper() == 'H' or atom_name.startswith('H'))
                and atom_name not in ('H', 'HN', 'H1', 'H2', 'H3')):
                continue
            if key in moved: line = line[:30] + moved[key] + line[54:]
        output.append(line)
    return '\n'.join(output) + '\n'

---

## 1 · Assess the pocket

Before docking, inspect the c-Abl structure and compare the known ligand site with predicted pockets in [ProteinsPlus](https://proteins.plus/). Record the pocket location, approximate size, enclosure, hydrophobic character, polar features and any concerns about missing residues or cofactors. Choose one plausible pocket for the focused search. A pocket prediction alone does not establish druggability.

Use the default focused box for the worked c-Abl example. Later, change to **Blind whole protein** and compare where poses are placed, the search time and whether the known site is recovered.

---

## 2 · Choose inputs

Set **Demo** for the worked c-Abl example. For your own protein, use **Upload ligand**, **SMILES**, or **Sketcher**. Upload a protein `.pdb` separately. Ligand files can be `.sdf`, `.mol2`, `.mol`, or `.smi`. A ligand PDB is not accepted because bond orders are often missing. The demo ligand starts from an imatinib SMILES and is embedded in 3D. Its input coordinates are not the experimental binding pose.

In [ ]:
# @title Step 3. Sketch a ligand (optional) { display-mode: "form" }
# Run this optional step before choosing Sketcher in Step 4.
# Draw one small molecule, then click Use this drawing.
show_sketcher = False # @param {type:'boolean'}
drawn_smiles = globals().get('drawn_smiles', '')
if show_sketcher:
    from google.colab import output
    output.enable_custom_widget_manager()
    import traitlets
    # ipyketcher 0.1.0 uses a subscripted traitlets.List at import time.
    # Some Colab traitlets builds do not implement this typing convenience.
    try:
        traitlets.List[str]
    except TypeError:
        traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)
    from ipyketcher import KetcherEditor
    import ipywidgets as widgets

    sketcher = KetcherEditor(return_formats=['smiles'])
    use_drawing = widgets.Button(description='Use this drawing', button_style='success')
    sketch_status = widgets.Output()

    def capture_drawing(_):
        global drawn_smiles
        with sketch_status:
            sketch_status.clear_output(wait=True)
            candidate = (sketcher.smiles or '').strip()
            if not candidate or Chem.MolFromSmiles(candidate) is None:
                print('Draw one valid molecule first, then click the button again.')
                return
            drawn_smiles = candidate
            print('Drawing saved. Select Sketcher in Step 4 and run that step.')

    use_drawing.on_click(capture_drawing)
    display(sketcher, use_drawing, sketch_status)
else:
    print('To draw a ligand, tick show_sketcher and run this step. Otherwise continue to Step 4.')

In [ ]:
# @title Step 4. Choose inputs { display-mode: "form" }
input_source = 'Demo' # @param ['Demo', 'Upload ligand', 'SMILES', 'Sketcher']
demo_ligand = 'Imatinib expected binder' # @param ['Imatinib expected binder', 'Caffeine comparison decoy']
ligand_smiles = '' # @param {type:'string'}

def embed_starting_conformer(source_mol):
    candidate = Chem.AddHs(source_mol, addCoords=True)
    if candidate.GetNumConformers() == 0 or not candidate.GetConformer().Is3D():
        candidate.RemoveAllConformers()
        params = AllChem.ETKDGv3()
        params.randomSeed = 17
        if AllChem.EmbedMolecule(candidate, params) != 0:
            raise ValueError('Could not make a 3D ligand conformer. Check the molecular structure.')
        if AllChem.MMFFHasAllMoleculeParams(candidate):
            AllChem.MMFFOptimizeMolecule(candidate)
        print('Generated a 3D starting conformer from the 2D ligand. Check stereochemistry and protonation.')
    return candidate

def read_ligand_file(path):
    suffix = path.suffix.lower()
    if suffix == '.sdf':
        molecules = [item for item in Chem.SDMolSupplier(str(path), removeHs=False) if item is not None]
        if len(molecules) != 1:
            raise ValueError('Provide an SDF with exactly one ligand molecule.')
        return molecules[0]
    if suffix == '.mol2':
        return Chem.MolFromMol2File(str(path), removeHs=False)
    if suffix == '.mol':
        return Chem.MolFromMolFile(str(path), removeHs=False)
    if suffix == '.smi':
        first = next((line.strip() for line in path.read_text().splitlines() if line.strip()), '')
        return Chem.MolFromSmiles(first.split()[0]) if first else None
    raise ValueError('Ligand file must be .sdf, .mol2, .mol, or .smi')

if input_source == 'Demo':
    receptor = WORK / '1fpu_receptorH.pdb'
    url = 'https://raw.githubusercontent.com/ccsb-scripps/AutoDock-Vina/develop/example/flexible_docking/data/1fpu_receptorH.pdb'
    urllib.request.urlretrieve(url, receptor)
    demo_smiles = {
        'Imatinib expected binder': 'CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5',
        'Caffeine comparison decoy': 'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
    }
    smiles = demo_smiles[demo_ligand]
    ligand_name = demo_ligand
    mol = Chem.MolFromSmiles(smiles)
else:
    from google.colab import files
    print('Upload the protein PDB')
    up = files.upload()
    if len(up) != 1 or not next(iter(up)).lower().endswith('.pdb'):
        raise ValueError('Upload exactly one protein .pdb file')
    receptor = WORK / 'protein.pdb'; receptor.write_bytes(next(iter(up.values())))
    if input_source == 'Upload ligand':
        print('Upload one ligand .sdf, .mol2, .mol, or .smi file')
        up = files.upload()
        if len(up) != 1:
            raise ValueError('Upload exactly one ligand file')
        filename, data = next(iter(up.items()))
        ligand = WORK / ('ligand' + Path(filename).suffix.lower())
        ligand.write_bytes(data)
        mol = read_ligand_file(ligand)
        ligand_name = Path(filename).stem
    else:
        smiles = ligand_smiles.strip() if input_source == 'SMILES' else globals().get('drawn_smiles', '')
        if not smiles:
            raise ValueError('Enter a SMILES in this form or save a drawing in Step 3 first')
        mol = Chem.MolFromSmiles(smiles)
        ligand_name = 'SMILES input' if input_source == 'SMILES' else 'Sketcher input'

atoms = [line for line in receptor.read_text().splitlines() if line.startswith('ATOM  ')]
if not atoms: raise ValueError('The protein PDB contains no ATOM records')
if mol is None: raise ValueError('The ligand could not be read. Check its format and chemistry.')
if mol.GetNumAtoms() == 0: raise ValueError('The ligand contains no atoms')
mol = embed_starting_conformer(mol)
prepared_sdf = WORK / 'ligand_prepared.sdf'
writer = Chem.SDWriter(str(prepared_sdf)); writer.write(mol); writer.close()
print(f'Protein: {receptor.name} ({len(atoms)} ATOM records). Ligand: {ligand_name} ({mol.GetNumHeavyAtoms()} heavy atoms)')

---

## 3 · Set the search and flexibility

**Focused (manual):** enter a box centre and lengths in Å, then adjust them visually in Step 6. **Blind:** the box encloses protein ATOM coordinates with the chosen margin. You can also adjust this box in Step 6. Blind search is much slower and is only a first-pass site search. Narrow the box around a plausible pocket for follow-up. Side-chain flexibility uses Meeko residue IDs such as `A:315` or `A:286,359`. The backbone stays fixed.

In [ ]:
# @title Step 5. Configure search { display-mode: "form" }
box_mode = 'Focused (manual)' # @param ['Focused (manual)', 'Blind (whole protein)']
center_x = 15.190 # @param {type:'number'}
center_y = 53.903 # @param {type:'number'}
center_z = 16.917 # @param {type:'number'}
size_x = 20.0 # @param {type:'number'}
size_y = 20.0 # @param {type:'number'}
size_z = 20.0 # @param {type:'number'}
blind_margin_A = 5.0 # @param {type:'number'}
flexible_residues = '' # @param {type:'string'}
exhaustiveness = 8 # @param {type:'integer'}
num_poses = 9 # @param {type:'integer'}
seed = 17 # @param {type:'integer'}

if box_mode.startswith('Blind'):
    xyz = np.array([[float(line[i:i+8]) for i in (30,38,46)] for line in atoms])
    lo, hi = xyz.min(0)-blind_margin_A, xyz.max(0)+blind_margin_A
    center, size = (lo+hi)/2, hi-lo
else:
    center = np.array([center_x,center_y,center_z], float)
    size = np.array([size_x,size_y,size_z], float)
if not np.isfinite(center).all() or not np.isfinite(size).all() or (size <= 0).any():
    raise ValueError('Box centre and lengths must be finite. Lengths must be positive')
if exhaustiveness < 1 or num_poses < 1: raise ValueError('Exhaustiveness and number of poses must be positive')
flexible_residues = flexible_residues.strip()
if flexible_residues and not re.fullmatch(r'[A-Za-z0-9]?:[0-9A-Za-z,]+(?:,[A-Za-z0-9]:[0-9A-Za-z,]+)*', flexible_residues):
    raise ValueError('Use Meeko residue syntax, for example A:315 or A:286,359')
print('Centre:', center.round(2), 'Å | size:', size.round(2), 'Å | volume:', round(np.prod(size)), 'Å³')
if np.prod(size) > 27000: print('Large box: consider higher exhaustiveness and repeat runs, then focus on candidate pockets.')

In [ ]:
# @title Step 6. Adjust the 3D box { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import clear_output

# The sliders directly update center and size, which Step 7 passes to Vina.
protein_xyz = np.array([[float(line[i:i+8]) for i in (30,38,46)] for line in atoms])
protein_lo, protein_hi = protein_xyz.min(0), protein_xyz.max(0)
axis_names = ('X', 'Y', 'Z')
axis_colors = ('#e34b4b', '#24a66a', '#327bd3')
box_controls = []
for i, axis in enumerate(axis_names):
    low = min(float(protein_lo[i]-20), float(center[i]-20))
    high = max(float(protein_hi[i]+20), float(center[i]+20))
    box_controls.append(widgets.FloatSlider(
        value=float(center[i]), min=low, max=high, step=0.5,
        description=f'{axis} centre', readout_format='.1f', continuous_update=False,
        style={'description_width':'75px'}, layout=widgets.Layout(width='340px')))
for i, axis in enumerate(axis_names):
    box_controls.append(widgets.FloatSlider(
        value=float(size[i]), min=2.0, max=max(100.0, float(size[i]*1.5)), step=0.5,
        description=f'{axis} size', readout_format='.1f', continuous_update=False,
        style={'description_width':'75px'}, layout=widgets.Layout(width='340px')))

box_summary = widgets.HTML()
box_preview = widgets.Output()

def redraw_box(change=None):
    global center, size, box_mode
    new_center = np.array([control.value for control in box_controls[:3]], float)
    new_size = np.array([control.value for control in box_controls[3:]], float)
    if not np.array_equal(new_center, center) or not np.array_equal(new_size, size):
        box_mode = 'Adjusted in 3D preview'
    center, size = new_center, new_size
    box_summary.value = (
        '<b>Search box used for docking</b> &nbsp; '
        f'centre ({center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}) Å &nbsp; '
        f'size ({size[0]:.1f}, {size[1]:.1f}, {size[2]:.1f}) Å &nbsp; '
        f'volume {np.prod(size):,.0f} Å³'
    )
    with box_preview:
        clear_output(wait=True)
        view = py3Dmol.view(width=850, height=540)
        view.addModel(receptor.read_text(), 'pdb')
        view.setStyle({'cartoon': {'color': '#9aabba'}})
        add_residue_hover(view)
        low_corner, high_corner = center-size/2, center+size/2
        point = lambda values: dict(zip('xyz', map(float, values)))
        view.addBox({'center': point(center),
                     'dimensions': dict(zip(('w','h','d'), map(float, size))),
                     'color': '#f07839', 'opacity': 0.11})
        edge_radius = max(0.10, min(0.24, float(np.min(size))*0.009))
        for i in range(3):
            for j in (0, 1):
                for k in (0, 1):
                    start = low_corner.copy()
                    start[(i+1)%3] = (low_corner, high_corner)[j][(i+1)%3]
                    start[(i+2)%3] = (low_corner, high_corner)[k][(i+2)%3]
                    end = start.copy()
                    end[i] = high_corner[i]
                    view.addCylinder({'start':point(start), 'end':point(end),
                                      'radius':edge_radius, 'color':'#e65f28'})
        axis_origin = low_corner - np.maximum(2.0, np.minimum(size*0.12, 5.0))
        axis_length = max(4.0, min(8.0, float(np.min(size))*0.32))
        for i, (axis, color) in enumerate(zip(axis_names, axis_colors)):
            tip = axis_origin.copy()
            tip[i] += axis_length
            view.addArrow({'start':point(axis_origin), 'end':point(tip),
                           'radius':0.23, 'color':color})
            view.addLabel(axis, {'position':point(tip), 'fontColor':color,
                                 'backgroundColor':'white', 'fontSize':16,
                                 'borderColor':color, 'borderThickness':1})
        view.zoomTo()
        add_snapshot_button(view, 'docklab_search_box.png')
        view.show()

for control in box_controls:
    control.observe(redraw_box, names='value')
display(widgets.HTML('<b>Move the box</b> by changing its centre. <b>Resize it</b> by changing its X, Y and Z lengths. Drag a slider and release to update the preview.'))
display(widgets.HBox([widgets.VBox(box_controls[:3]), widgets.VBox(box_controls[3:])]))
display(widgets.HTML('<span style="color:#e34b4b">● X</span> &nbsp; <span style="color:#24a66a">● Y</span> &nbsp; <span style="color:#327bd3">● Z</span> &nbsp; Orange is the docking box. Rotate and zoom the 3D view with your mouse.'))
display(box_summary, box_preview)
redraw_box()

---

## 4 · Prepare and dock

Meeko prepares ligand and receptor PDBQT files. If preparation stops, fix the source structure rather than treating an incomplete receptor as a valid model. The demo permits unresolved residues because the official Vina tutorial uses that option for 1FPU. Check the preparation log before interpreting that run.

In [ ]:
# @title Step 7. Run docking { display-mode: "form" }
rec_base = WORK / 'receptor'
cmd = ['mk_prepare_receptor.py', '--read_pdb', receptor, '-o', rec_base, '-p', '-j']
if flexible_residues: cmd += ['-f', flexible_residues]
if input_source == 'Demo': cmd += ['--allow_bad_res']
preparation = run(*cmd)
if 'Template matching failed' in preparation.stdout:
    print('Some incomplete residues were omitted from the docking receptor. Inspect preparation_log.txt in the results.')
(WORK/'preparation_log.txt').write_text(preparation.stdout + preparation.stderr)
run('mk_prepare_ligand.py', '-i', prepared_sdf, '-o', WORK/'ligand.pdbqt')
rigid = WORK / ('receptor_rigid.pdbqt' if flexible_residues else 'receptor.pdbqt')
flex = WORK / 'receptor_flex.pdbqt'
if not rigid.exists(): raise FileNotFoundError(f'Expected {rigid.name}. Inspect Meeko output')
if flexible_residues and not flex.exists(): raise FileNotFoundError('Flexible residues were not prepared. Check residue IDs')
cmd = [VINA_BIN, '--receptor', rigid, '--ligand', WORK/'ligand.pdbqt',
       '--center_x', center[0], '--center_y', center[1], '--center_z', center[2],
       '--size_x', size[0], '--size_y', size[1], '--size_z', size[2],
       '--exhaustiveness', exhaustiveness, '--num_modes', num_poses, '--energy_range', 10,
       '--seed', seed, '--cpu', 2, '--out', WORK/'poses.pdbqt']
if flexible_residues: cmd += ['--flex', flex]
docking = run(*cmd)
(WORK/'docking_log.txt').write_text(docking.stdout + docking.stderr)
(WORK/'run_settings.json').write_text(json.dumps({
    'ligand_name': ligand_name, 'box_mode': box_mode, 'center_A': center.tolist(), 'size_A': size.tolist(),
    'flexible_residues': flexible_residues, 'exhaustiveness': exhaustiveness,
    'num_poses_requested': num_poses, 'energy_range_kcal_mol': 10, 'seed': seed}, indent=2))
run('mk_export.py', WORK/'poses.pdbqt', '-s', WORK/'poses.sdf')
for old in WORK.glob('pose_[0-9]*.sdf'):
    old.unlink()
for old in WORK.glob('pose_receptor_*.pdb'):
    old.unlink()
if flexible_residues:
    pose_blocks = re.findall(r'(?ms)^MODEL\s+\d+\s*\n(.*?)^ENDMDL', (WORK/'poses.pdbqt').read_text())
    if not pose_blocks: raise ValueError('Could not find pose models for flexible receptor export')
    output = []
    for pose_i, block in enumerate(pose_blocks, 1):
        pose_protein = flexible_protein_from_pose(block, receptor)
        (WORK/f'pose_receptor_{pose_i}.pdb').write_text(pose_protein)
        output.append(f'MODEL     {pose_i}')
        output.extend(pose_protein.splitlines())
        output.append('ENDMDL')
    (WORK/'poses_receptor.pdb').write_text('\n'.join(output) + '\n')
for old in WORK.glob('minimised_pose_*'):
    old.unlink()
for old in WORK.glob('minimised_receptor_pose_*'):
    old.unlink()
summary_path = WORK/'minimisation_summary.csv'
if summary_path.exists(): summary_path.unlink()
print('Docking complete. Poses are ready to explore below.')
pose_results = [tuple(map(float, match)) for match in re.findall(
    r'^REMARK VINA RESULT:\s+(-?[0-9.]+)\s+([0-9.]+)\s+([0-9.]+)',
    (WORK/'poses.pdbqt').read_text(), re.M)]
if not pose_results: raise ValueError('Vina wrote no scored poses. Inspect docking_log.txt')
with (WORK/'pose_scores.csv').open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['ligand', 'pose', 'vina_score_kcal_mol', 'rmsd_lower_A', 'rmsd_upper_A'])
    for i, (score, lower, upper) in enumerate(pose_results, 1):
        writer.writerow([ligand_name, i, score, lower, upper])
table_rows = ''.join(
    f'<tr><td>{i}</td><td>{score:.2f}</td><td>{lower:.2f}</td><td>{upper:.2f}</td></tr>'
    for i, (score, lower, upper) in enumerate(pose_results, 1))
display(HTML(f'<h3>Docked pose summary: {ligand_name}</h3><table style="border-collapse:collapse;width:600px">'
             '<thead><tr style="background:#e8f0f5"><th>Pose</th><th>Vina score (kcal/mol)</th>'
             '<th>RMSD lower (Å)</th><th>RMSD upper (Å)</th></tr></thead>'
             f'<tbody>{table_rows}</tbody></table>'
             '<p>RMSD values are relative to the best-ranked pose. Scores are estimates, not measured affinities.</p>'))

In [ ]:
# @title Step 8. Explore docked poses { display-mode: "form" }
scores = [row[0] for row in pose_results]
poses = [m for m in Chem.SDMolSupplier(str(WORK/'poses.sdf'), removeHs=False) if m is not None]
if not poses: raise ValueError('Meeko did not export readable SDF poses')
for pose_i, pose_mol in enumerate(poses, 1):
    pose_mol.SetProp('ligand_name', ligand_name)
    pose_mol.SetIntProp('pose_number', pose_i)
    pose_writer = Chem.SDWriter(str(WORK/f'pose_{pose_i}.sdf'))
    pose_writer.write(pose_mol)
    pose_writer.close()
import ipywidgets as widgets
from IPython.display import clear_output

if flexible_residues:
    receptor_models = re.findall(r'(?ms)^MODEL\s+\d+\s*\n(.*?)^ENDMDL',
                                 (WORK/'poses_receptor.pdb').read_text())
    if len(receptor_models) != len(poses):
        raise ValueError('Flexible receptor model count does not match ligand pose count')
else:
    receptor_models = [receptor.read_text()] * len(poses)

options = [(f'Pose {i+1}  |  {scores[i]:.2f} kcal/mol' if i < len(scores) else f'Pose {i+1}', i)
           for i in range(len(poses))]
pose_menu = widgets.Dropdown(options=options, value=0, description='Show pose:',
                             style={'description_width': 'initial'}, layout=widgets.Layout(width='340px'))
protein_menu = widgets.Dropdown(options=['Cartoon + pocket', 'Cartoon', 'Sticks', 'Ball and stick', 'Lines'],
                                value='Cartoon + pocket', description='Protein:',
                                style={'description_width': 'initial'}, layout=widgets.Layout(width='250px'))
ligand_menu = widgets.Dropdown(options=['Sticks', 'Ball and stick', 'Lines'], value='Sticks',
                               description='Ligand:', style={'description_width': 'initial'},
                               layout=widgets.Layout(width='220px'))
pose_output = widgets.Output()

def show_pose(change=None):
    i = pose_menu.value
    with pose_output:
        clear_output(wait=True)
        view = py3Dmol.view(width=850, height=500)
        view.addModel(receptor_models[i], 'pdb')
        add_residue_hover(view)
        view.addModel(Chem.MolToMolBlock(poses[i]), 'sdf')
        style_protein(view, protein_menu.value, receptor_models[i], poses[i])
        style_ligand(view, ligand_menu.value)
        view.zoomTo({'model': 1})
        view.zoom(0.8)
        add_snapshot_button(view, f'docklab_pose_{i+1}.png')
        view.show()

pose_menu.observe(show_pose, names='value')
protein_menu.observe(show_pose, names='value')
ligand_menu.observe(show_pose, names='value')
display(widgets.VBox([widgets.HBox([pose_menu, protein_menu, ligand_menu]), pose_output]))
show_pose()
print(f'{len(poses)} pose(s) available. Change the pose or representation above. Hover over visible pocket atoms or choose Lines for atom-level inspection across the protein. Use Save current view as PNG after rotating or zooming.')

### Compare an expected binder with a decoy

Run the focused search first with **Imatinib expected binder**. Record the best score, pose location and main contacts, then download the results. Return to Step 4, choose **Caffeine comparison decoy**, and rerun Steps 4 to 10 with the same box and search settings.

Compare the best scores and use ContactCanvas to compare the number and types of contacts. You should normally find that imatinib makes a richer interaction network and receives a more favourable score in this pocket. Treat this as a controlled teaching comparison. One score or interaction count cannot establish activity, and caffeine is not a validated inactive control for c-Abl.

### Optional local energy minimisation

Choose one docked pose below and click **Minimise selected pose**. Vina locally adjusts the ligand position, orientation and rotatable bonds using its docking scoring function. If you selected flexible side chains in Step 5, those side chains can move too. The rest of the protein stays fixed. Docking already includes local optimisation, so this extra step may make only a small change. The score may not improve. This is not an Amber minimisation of the full protein–ligand complex, and a Vina score is not a measured binding free energy.

In [ ]:
# @title Step 9. Minimise a pose (optional) { display-mode: "form" }
minimise_menu = widgets.Dropdown(options=options, value=0, description='Choose pose:',
                                 style={'description_width':'initial'},
                                 layout=widgets.Layout(width='350px'))
minimise_button = widgets.Button(description='Minimise selected pose', button_style='success',
                                 layout=widgets.Layout(width='210px'))
minimisation_output = widgets.Output()

def vina_score_from_log(log):
    match = re.search(r'Estimated Free Energy of Binding\s*:\s*([-+]?[0-9]+(?:\.[0-9]+)?)', log)
    if not match:
        raise ValueError('Vina did not report a score. Inspect the minimisation log.')
    return float(match.group(1))

def minimise_selected_pose(_):
    pose_i = minimise_menu.value
    pose_number = pose_i + 1
    minimise_button.disabled = True
    with minimisation_output:
        clear_output(wait=True)
        print(f'Minimising pose {pose_number}...')
        try:
            blocks = re.findall(r'(?ms)^MODEL\s+\d+\s*\n(.*?)^ENDMDL',
                                (WORK/'poses.pdbqt').read_text())
            if pose_i >= len(blocks): raise ValueError('The selected pose is missing from poses.pdbqt')
            ligand_block, marker, flexible_block = blocks[pose_i].partition('BEGIN_RES')
            ligand_input = WORK/f'minimised_pose_{pose_number}_input.pdbqt'
            ligand_input.write_text(ligand_block.strip() + '\n')
            common = [VINA_BIN, '--receptor', rigid, '--ligand', ligand_input,
                      '--center_x', center[0], '--center_y', center[1], '--center_z', center[2],
                      '--size_x', size[0], '--size_y', size[1], '--size_z', size[2], '--cpu', 2]
            if flexible_residues:
                if not marker: raise ValueError('The selected pose has no flexible side-chain coordinates')
                flexible_input = WORK/f'minimised_pose_{pose_number}_flex_input.pdbqt'
                flexible_input.write_text(marker + flexible_block.strip() + '\n')
                common += ['--flex', flexible_input]
            before_log = run(*common, '--score_only').stdout
            before_score = vina_score_from_log(before_log)
            refined_pdbqt = WORK/f'minimised_pose_{pose_number}.pdbqt'
            result = run(*common, '--local_only', '--out', refined_pdbqt)
            (WORK/f'minimised_pose_{pose_number}.log').write_text(
                'Before minimisation\n' + before_log + '\nAfter minimisation\n' + result.stdout)
            after_score = vina_score_from_log(result.stdout)
            # Vina retains the original docking energy remarks in local-only output.
            # Remove them so the exported files never show a stale score.
            stale_energy = ('REMARK VINA RESULT', 'REMARK INTER', 'REMARK INTRA', 'REMARK UNBOUND')
            refined_pdbqt.write_text('\n'.join(
                line for line in refined_pdbqt.read_text().splitlines()
                if not line.startswith(stale_energy)) + '\n')
            refined_sdf = WORK/f'minimised_pose_{pose_number}.sdf'
            run('mk_export.py', refined_pdbqt, '-s', refined_sdf)
            refined_molecules = [m for m in Chem.SDMolSupplier(str(refined_sdf), removeHs=False)
                                 if m is not None]
            if len(refined_molecules) != 1:
                raise ValueError('Meeko did not export exactly one minimised ligand pose')
            refined_mol = refined_molecules[0]
            refined_mol.SetProp('vina_score_after_minimisation_kcal_mol', f'{after_score:.3f}')
            refined_mol.SetProp('source_docked_pose', str(pose_number))
            sdf_writer = Chem.SDWriter(str(refined_sdf))
            sdf_writer.write(refined_mol)
            sdf_writer.close()
            if flexible_residues:
                refined_protein = flexible_protein_from_pose(refined_pdbqt.read_text(), receptor)
                (WORK/f'minimised_receptor_pose_{pose_number}.pdb').write_text(refined_protein)
            else:
                refined_protein = receptor.read_text()

            summary_path = WORK/'minimisation_summary.csv'
            fields = ['pose', 'before_vina_score_kcal_mol', 'after_vina_score_kcal_mol',
                      'score_change_kcal_mol', 'method']
            previous = {}
            if summary_path.exists():
                with summary_path.open(newline='') as handle:
                    previous = {int(row['pose']):row for row in csv.DictReader(handle)}
            previous[pose_number] = dict(pose=pose_number,
                before_vina_score_kcal_mol=f'{before_score:.3f}',
                after_vina_score_kcal_mol=f'{after_score:.3f}',
                score_change_kcal_mol=f'{after_score-before_score:+.3f}',
                method='AutoDock Vina local optimisation')
            with summary_path.open('w', newline='') as handle:
                writer = csv.DictWriter(handle, fieldnames=fields)
                writer.writeheader()
                writer.writerows(previous[i] for i in sorted(previous))

            display(HTML(
                f'<h4>Pose {pose_number}: Vina local minimisation</h4>'
                '<table style="border-collapse:collapse;width:510px">'
                '<tr><th>Before</th><th>After</th><th>Change</th></tr>'
                f'<tr><td>{before_score:.3f}</td><td>{after_score:.3f}</td>'
                f'<td>{after_score-before_score:+.3f} kcal/mol</td></tr></table>'
                '<p>The before score is recalculated for this pose and may differ slightly from the docking table. '
                'These are Vina scores, not measured binding free energies.</p>'))
            comparison_menu = widgets.ToggleButtons(
                options=['Before minimisation', 'After minimisation'],
                value='After minimisation', description='View:',
                style={'description_width':'initial'})
            comparison_output = widgets.Output()

            def show_comparison(change=None):
                after = comparison_menu.value == 'After minimisation'
                protein_text = refined_protein if after else receptor_models[pose_i]
                ligand_mol = refined_mol if after else poses[pose_i]
                with comparison_output:
                    clear_output(wait=True)
                    view = py3Dmol.view(width=850, height=500)
                    view.addModel(protein_text, 'pdb')
                    add_residue_hover(view)
                    view.addModel(Chem.MolToMolBlock(ligand_mol), 'sdf')
                    style_protein(view, 'Cartoon + pocket', protein_text, ligand_mol)
                    style_ligand(view, 'Sticks')
                    view.zoomTo({'model':1})
                    view.zoom(0.8)
                    suffix = 'after' if after else 'before'
                    add_snapshot_button(view, f'docklab_pose_{pose_number}_{suffix}_minimisation.png')
                    view.show()

            comparison_menu.observe(show_comparison, names='value')
            display(comparison_menu, comparison_output)
            show_comparison()
            print(f'Saved {refined_sdf.name}. Download it in Step 10 for ContactCanvas.')
        except Exception as exc:
            print(f'Minimisation failed: {exc}')
        finally:
            minimise_button.disabled = False

minimise_button.on_click(minimise_selected_pose)
display(widgets.HBox([minimise_menu, minimise_button]), minimisation_output)
print('This step is optional. Choose a pose and click the button when you want to minimise it.')

In [ ]:
# @title Step 10. Download results { display-mode: "form" }
bundle = WORK/'docklab_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(receptor, 'protein_input.pdb')
    for name in ['poses.pdbqt','poses.sdf','pose_scores.csv','run_settings.json','preparation_log.txt','docking_log.txt','receptor.json','receptor.pdbqt','receptor_rigid.pdbqt','receptor_flex.pdbqt','poses_receptor.pdb','minimisation_summary.csv']:
        p=WORK/name
        if p.exists(): z.write(p, name)
    for p in sorted(WORK.glob('minimised_pose_*')):
        if p.suffix in ('.pdbqt', '.sdf', '.log') and not p.name.endswith(('_input.pdbqt', '_flex_input.pdbqt')):
            z.write(p, p.name)
    for p in sorted(WORK.glob('minimised_receptor_pose_*.pdb')):
        z.write(p, p.name)
    for p in sorted(WORK.glob('pose_[0-9]*.sdf')):
        z.write(p, p.name)
    for p in sorted(WORK.glob('pose_receptor_*.pdb')):
        z.write(p, p.name)
print('Ready:', bundle)
from google.colab import files
files.download(str(bundle))

### Compare interactions before and after minimisation

Open **ContactCanvas** and upload the protein and ligand separately. For a rigid run, pair `protein_input.pdb` with the exported `pose_N.sdf` that you minimised. Then repeat with `protein_input.pdb` and `minimised_pose_N.sdf`. Compare the residue list, interaction types and geometry before and after minimisation.

For a flexible run, pair `pose_receptor_N.pdb` with `pose_N.sdf`, then pair `minimised_receptor_pose_N.pdb` with the corresponding minimised ligand.

Look for small rearrangements rather than expecting a dramatic change. Docking already includes local optimisation. A changed contact map can reflect geometry thresholds and does not prove that the refined pose is biologically correct.

### Optional ligand-based virtual screening

If time permits, open [SwissSimilarity](https://www.swisssimilarity.ch/). Use imatinib or another lead compound as the query, choose a small appropriate library, and compare a 2D method with a 3D or combined method. Record the query representation, library, method, similarity score and two candidates you would investigate next. Similarity suggests candidates. It does not establish binding or activity.

### References

1. Trott O, Olson AJ. AutoDock Vina: improving the speed and accuracy of docking with a new scoring function, efficient optimization, and multithreading. *J Comput Chem.* 2010, 31:455–461. [doi:10.1002/jcc.21334](https://doi.org/10.1002/jcc.21334).
2. Eberhardt J, Santos-Martins D, Tillack AF, Forli S. AutoDock Vina 1.2.0: new docking methods, expanded force field, and Python bindings. *J Chem Inf Model.* 2021, 61:3891–3898. [doi:10.1021/acs.jcim.1c00203](https://doi.org/10.1021/acs.jcim.1c00203).
3. Forli Lab. *Meeko* [software]. [GitHub repository](https://github.com/forlilab/Meeko).
4. Rego N, Koes D. 3Dmol.js: molecular visualization with WebGL. *Bioinformatics.* 2015, 31:1322–1324. [doi:10.1093/bioinformatics/btu829](https://doi.org/10.1093/bioinformatics/btu829).
5. RDKit contributors. *RDKit: Open-source cheminformatics* [software]. [doi:10.5281/zenodo.591637](https://doi.org/10.5281/zenodo.591637).
6. EPAM Systems. *Ketcher: web-based chemical structure editor* [software]. [GitHub repository](https://github.com/epam/ketcher).
7. Smit J. *ipyketcher: Ketcher editor as anywidget* [software]. [PyPI package](https://pypi.org/project/ipyketcher/).
8. Zoete V, Daina A, Bovigny C, Michielin O. SwissSimilarity: a web tool for low to ultra high throughput ligand-based virtual screening. *J Chem Inf Model.* 2016, 56:1399–1404. [doi:10.1021/acs.jcim.6b00174](https://doi.org/10.1021/acs.jcim.6b00174).